# OpenPlaque V24 — native coronary candidate extraction (research only)

Compares V22 raw independent LM/LAD/LCX/RCA predictions with V23's **10 mm CT-derived heart/aorta region** and additive CT attenuation, bright-tubular Frangi, component shape, and aorta-contact proxy screens. No neural model training, no annotation-informed proposal generation, no sealed test reads. Unannotated voxels remain **unknown**, not verified negatives. Aorta contact is NOT confirmed ostial origin.

**Run:** Select a Colab runtime with adequate memory, then Runtime → Run all. First cell mounts Drive. Next cell has reuse/full-cohort controls. A pinned 12-case pilot stops the run if fewer than 90% of V22-known-positive hits are retained for any artery, or if fewer than 85% of the raw V22 prediction voxels are removed at the lower Frangi threshold. These are exploratory thresholds, not clinical acceptance criteria. The full 80 runs only after the pilot passes. Each completed case is SHA-validated and reused.

This notebook packs its standalone UTF-8 evaluator in a compressed source cell to preserve one-click launch and immutable checksums. The following cell extracts the readable Python file to /content/openplaque_v24_candidates.py before execution.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cache / run switches (leave frozen cohort and predeclared gate unchanged).
RUN_SYNTHETIC_TESTS = True
RUN_PILOT = True
RUN_ALL_80_ONLY_IF_PILOT_PASSES = True
REUSE_VALID_CACHES = True
print('V24: CT-derived candidate screens, unchanged V22 and V23 source files')

In [ ]:
import subprocess,sys,importlib.util
for module,pkg in [('nibabel','nibabel'),('scipy','scipy'),('skimage','scikit-image')]:
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable,'-m','pip','install','-q',pkg])
print('Ready: native NIfTI, scipy, 3D bright-vesselness')

In [ ]:
import base64,zlib,hashlib
from pathlib import Path
PACKED_SOURCE = 'eNrdfNty47iS4Lu/gl0d0yTLNHXzlTYdobbdp2rGZdfYrp4+oVUwKBGy2KZIHpJS2eX2xH7E/su+76fsl2xmAiBAXVyuc962oluWCCCRyHsmAL579+46Z+nnJPzHnBm/d3eNHePsbidLkydjwcqSJcY4TKM4CitmlOOCsbQ0shR6do1JNi92piyMjPOL3w/bRl6wKB5XcZaW7tbFY55kRVhlxZNRZpPqa1gwo2AlC4vx9NhIM6MqwjiN03vHSMIRzAODK/oJfRIWGRUrKyMcjwELx8gKY5xA93EICCVhPIMp3r17txXP8qyAbsV9HhYlc4xpWE6TeOQYf5ZZCuNgLEw0ZqNw/OAY9+OtSZHNjDyssJchhn+Gn7wB11nFMyZb5G+AAp/fspTxfuMsSRhfq+x6ls3TihVbW79f3Nx+vL7yzQwomxNldxbd3Z2ILQ7bO9V8NE/CIq6edvIie4xn+K3b7u532p3eTsfcurm+vvMRI8tsjTMAmVatqIgXrPXp6Zz+Ko6Z9tav/dsLH8e0TPU8uFiEyTxE/ILzsApLVgWLdtA1t4BxPg5pmUx1WXS7wWSeJMEiS+Yzhr1663r1AmB3UQUhrDcMiiw2t66/3K3ruRtw6Qlq6SnNrbO7W4HpxSNQKg0TxK31cRbes7P+bXBdxPcxPD07u+sHn8JqPGXRXRbI9j+CRafFMQRgNxe/SWhnIfRUYP4IPheMpItFOEQ27PwRRJwW7rc4D9gjCkbFojUdWiW7nwHhaTkwGQi4WGWccswD4GZw2A5AGr6x1K0eK+oV3H7o+yabdEdRbxSN2EE0PhqPJrv7vSO2340O2xGI4l6v0z4Mx4e9/XA8mezv7ncOWHvSORjtTrpHI7Y7AVJ9uDj7j8/XH6/uOMTJweERYxHA6Bx29nd7ve5uuwsfe3vRQXhwuN8dH7BwMt4/Co8OjsK99lHvkB30ertH4f7B0f7oyNz6cNE/v/WfzctPpmd1nLbtmJf9c/jedTr4/ewP+N5zuvD95qwP34+cnv2y9fnjJYijZXY6pmN2drvw2W334LPX3sPPgzZ8AhrwubePrfs9bD1oH+LnwT58Hu7hkyP4bm+dX//X1W3/0+fLCx8mw+kM+PczKDBoT2R0dmYzIy6zqsjyeHwMaoZaDYYnK0AFU2DHghmzsHwowfoUxgzYHI9LUJmPwadPfqftgowFnz5eBR+++J2u+Nn/g37utuH3bzf9q799DG4//u1T/9a32u6R03H3na67Z4Pe3t5eXAZ3H24ubj9cX55Te7vrwMe+vXX35deL4PzjZf8OdJtmc3tbwJvPFzThp089/8DVHvz66/Uf2E09BDwu/vPLx98BSv8TNh25pOzw+/auf3V2gc8O3S0C1//365vg7Izg4rqID8Hf+ncXwc3F3cUVYuED+u1my/mXM9FyuLd1e/fl/O/B7/2bj/0rUD3LBI3tIL+mc/oDpogFSSa/TWP1LCinYc6030WWVcC/ra2ITQxotHLb20LeTX1hcl142t3bt2x6/DWupmRXoaOLhhBmH5m2EQLn+ED8h0wcT+fpgxGnRgyaZSXhbBSF3sQtwLFYu+877S7/sJ2Radre1J3naE4sGsbnAm2fF6kxdafsMYrvwW9YElOCAqiKLugU3CQLo9KSuGGPoAJjYNk2H/MVjDOzcmchVpj7ou9x7oKXAbPgzh6iuLD4j9K/K+bMYY9xWQXZA/3ieFV+7iIdgjScAUAX/2ybbjXLTdHBpbn49IRbNJ/lpbVw4jQC0H7XKcHUBg/sScwSJkn2FeCl/m9hUjKbw8lKWAZ4mjGzKieXa4+jMgDrG8zCNJ4QUfh64Lk/eHRL0J3csokHj0h/MF86NdwyT+IKnC4roVc8MeohQwIDlF5mvWm6f2Zxaj1um/8jNRVkXASLLJjYgaX4YENt22XpOIsYUL3BNkJwYiQgMNDd/snHb2CR6ZeNcYBqgpgDfsPQn3xhevF3mlUGDiC9sE98OVhJHbgGMCo34K7Bp18URVZY5u3d9WeDG3MRzsTIAXDOxiwuZ+iMBNNK8KAkVuBKW+bXaZYw4TeDcj6bhcUT+lMX2VmP6MkRvZapOvX0TrMGWMmzZVCzBiitVwMWENACNN17WLo5zmZ5wiqIFcCyEnVQkpBSqgv4vDJQHQVpa4LVHWfAMfDpUzZ+yIHRVcAZjyOaHqs5usdHp6tTqMayAieMz02KROqeOqSZxAOdL4q3ml4TgJkEicva2E0D2vvBxf2AKCEzgVUGwaYoNptisAih34KBGo/ZsnhJnvrP9STmghUlBCKmJ0JLxzA5fIpCBJqeWJsj2xbdzprVeM21ONosEAWWENOPmem9Fh06JkaC63uuRojaBHmcZFUguWJ6CVhMoaaOaLwH0x4sIDwO0wpgKw+lNc/iNAB7zuM7IHCQZ2WMkQF4sKoMclYEgAIrnkxvnddcQUgDGs3HfCElAd7F7CZQeQ1Q4JEl5RJc4XM1uLBusLoFBLPBbGZ6PDwBpvEAJhhXwXQefAUbn301vUEdsjh1tDLUgPEYKIC+aRmiUgSPT98E8VQ4ZSPbw/Q+Dsr4HiKkQAwDjEXfRuhjaxOIcSOIvqdVUMQRmGLTI3dj8igeXEAZVFNI38Da1ZxbiZd0oPpAsJAFinoQxQkXFKTKckDVWDKkVCnjQpuzYBInwE+Y99lEPglrO5v1TK8RgTnUPBpljzBVeJ9hnI5TrURlaq5X/oF5fQzYP+YxCDjiAiAh4ATp0kAux3Qv2iqkDmQ4FMwbUoCLQzPmawgOKggkfZCYBDHIMmSIT0HKH07iYgYSn5VVPJ8JBqmh3HppOXiA+XmAgQREX+jLUDWjcmVgwSYM4hhEDgPrYF7iJGnyFIQTXG2dwQX3LAVGVmSJloCkWSDzeSk6+Ah9U6HNig+54QgA33gC+TzPsJbhzcE2wrJh1kgoHVEBDCEMgmcpuydd0gGHoF7ZDEsEgfTdSMBZtn4KYWWkNDeDZch8sBNkt2+M9XIf+jY88i73yNIh5y6NwlgKCCoD0598OcLb6EpoHBCWyjPZvMrnDf8BZJygctSuH128ms2T4aycqBEzY0gmG0TYOIbcK+D5mIUS4ZR5OMbCTB5GEfwFGfa7B64MJnnxI4W49QmD+zQXMeajn0L0OwXRIigN7KB10B66ZfyNrV32xSyHuOsO+J/cykQcfCe5FrFOQAYnGLM4sRRiLXgUFkX4ZAmkIboMy+opZxYGnds9GptkOBS0O57NZ1bbqQcNQhcMiIiJQ8pJosehvQMT8FmnMY0EOceRuC6XLFQTRPho2dudJSDbNRBB+moOttwqk3hMyFmh7eCfkS2md0Y49hsE3EnmTGNbhvXCiXBpfwLGVuBpWLWRH5A3xyOqsOHPeETdqIglnrsgTFhlQxETw8HSc1eD/cCAi8ikfPBhDCVQFuQCIL81WxUtfvI5QvyXjMiRQEkyTrKSy4MbTibgwAXy8hcsKfE7bKenhVg07TJCXDItfaYmKNvJiogVfrtBc8Si5IwiJLDck43+dCIUEWTtHFhwaJ+2ZfqICbNFcueQOZfKIPAjMpbjOH9ywUhiGakuHEqLD2YxLYGhs4BF1RavdvwHY7lRTZkBi4IRykQbyoYZMD/IGdouZBwEkY4R5pCSsQjlQtZCZi5Pxqb++hmt/+bYE/kAcV8uQOqFOQF2Vr2u0Kso3AhIEOCNgArfiqYnPo997L8sKwpP/EPX/kV7vN3punaDQYUThYL2mHkGSfzAMIiwpnMHmLFE/nfv3p3xCtFvFMMYPIYxMGwsXeMWy0YG0HI8T9CTGCM2yagMHaK9MGrPZwhnA7R2sar8imHjHH8gXrsiLpF9eSC1JBg1ICEfCCrinXiI5uuG4ArCHoc+0BaUwhaUlJ6rWI+P9jVpns4HHNpQybLgh6Dv2Af6yU5CDq+At2ECVtg4uzOwspmWmPBqApk8GdkEyPMEyBYQTxVPpI+o1RrNFD7jBOyVNd6pY1q7ZR212672wGk7HVsg0F9kcWRgwFPEozkvp/M6PMkgcDIuS0xW51UJPl1Xj4Ldq6kH/w00dUcxIlhHmVYxdrCmxEMMv2cP/TbvT/VEn7PLGjs8YvYbAbITJvk09N09ZwSpBv1NwvGDCI9F9UX3abD4uCSNBV4ifBsNHvjfdQ7uCqM57CoFl0YIzRGRuxB1IR7ETOnVBg9D+72SB/hJYkJVNITHrJ5QK7DSkNL4SB0RvbFAkIk3zcE/9yRDUk47hhUaS3rX5Ti9hS6yiaNdK3E5TyBIeubRE+LEswYs+0hvtpI2ONaaSqSteYAHMJc+UejU5/DqJiS+QNqjbuvkABucerk+/6aLhgRh13B/Ns7rfCsyynlOOswec4jdcG8qkwaYglPjnmVYi346NqKMpAGCaNIaZWLI39TwJSQfccPiHQv1bA6iIwfCkxLc15rGjmjsrGvsisauWoxgywCYMPTlzANvOueOEydTPzr6j+6Qm4o4nWT+s0xFeW2Yh81j3hPSz6ZIUKbDezRFRU+9ZOqZZuk3VmR1ei3FD3fUZKNQqrXjMVHjGuRxPeHhXT1E9suPjqgmoXr9Yw5WJk6Eyg6K8dBxj45sqncWYxdYCNEg4MQMNMovDVfFqeogbWToXOetlEpZXBDRcVEauM57iQG1BIlNVsoRyOHuiHTQoFwQBYzxKl6BqWnx477rO05JmDOOOieAp6In5ERJXlksznbWP30206AIv5peG3IzKTH0nafFtBp4wklKO8Clk5L60g9JO6W237dh0nzOMVEDtKAT/bA4eLcIFwDYdsCAQUJ/X039dLujDwJN8NucxzwR3wBpIDk6XIJ1rI1TwLhhEmSyoJ/w0aMsS4SZBp0saYXgFiD7Hv3JxpC88dl4F9INvxZcoF+kkpza3EKElKChZcB7opRFoIE4nm4zsVNJIu2heMXpnNXNC1/gHw/f06T6wMVJo+CyOpo9VnpcMrBKt6yyfAf/QBxqvy9zHtKAMkiPUCa1ZgyV1RJkrVdc/qPgfyHzglnev+9Kt0PzYp2GSjR+130vuy9aFtoBDsqBxGLXxrY81kby3QZoP/VX6kSkhgr0ib+h7OM16knIb6AeVQc4e1H8ydQPOEuHtZD5PEv6xbB00RG9TkX2Isyv0CiyjramVmvNJc62ZC2XdG/jsF90XAAHW1RDsITNMdPtIE++yL5pprBW0BDkGKtSDuDugDt0AILDkvg+HiXMgbBelhceUvC4QQZpUfF6beFNZuz76pIHKcU5iBwJlQ3qG34VT+HbwMSfq9VfIaMQkwJP1hSHAbCjP6c6JTzk2qRxBN0S79KcqAwKNssWLDK9zg4MbKEMdxxCTueomWRfP3wJgCHZVyxgz8teu72uXI0LonX+AomCcWLsQD/bFoteBSjRkn5y01CBF2DYhKIqiuMkLMtActv00OJZ8qeQKtA/+UTbV2GTPCt9C/76PkiMfZziV+IMNQnk6wGZODuimKcKiI39AVMzMfjbV8ThkO0V0PNKbjo/a2uroUoqI4bOug40rYef6+vOOsGyNKKcH9AuGOQ5CR8oSI1T2BuA1MIkSUFbGLQlIndLGrBkt03wdHkEqJp6iu2VgNsCXbx0HRZkfOGExBjK+x4lOaEo/X3bcigoszeFEGR2/pkAYpa/Fj/Yg443lCZgIHY1KYorg3sWdNqo8EN/rXkl2NwQnPrLBz1sW0FNsJhVVgq20sqhMGsEi1cbdU2U2+fULIJXkfcqA+5Qq27GYVYtip0DiXCjzhrH0T9RWfxu5aPCE3j+2d1ta2I+wxwvbjy7d9MYOPTNPEZFxHY841V3kAUI2YsbcWAxdaVdc93oYgBu1oMX9W6eKJ+CiWgAWjS233Pa9yiZAqDtri8ae/Cvdg1T8vHUscJydsnuJWrQV/ShQq7/jMdDPD4CYeHPF4kjxUwprxYalskr4Q7fYjKFIUU6lP6A09YRNHQkhZz3YiIX92vxTMdQj/exVoEVDNxrY6ICTgEaQZVVjN+g8SqrfsPNAVHKwE2JWcyrx0gHGIMbFOY2Sk59GmHxL50gWMgtemTjD48VMy8JhzaYjjMJMtn6WMx+YG70JUQ6OWY4QFYMxUgaJfnj4E+km6R2XLEZENt+y6GBeDabVyE4wvrcAFGU75gZuDB5WMDTCcwz0YD0gHahNsgkV716O0/1V89W1GY3aJwIa+gMRYk/CAXHNIBMwNbqYZTpEcWR0zqbhs7792tZUQcRXILrTbZagC2NPE5z9U69Ap09kIU34/mIq7sGx260o4BHQkDFSQ3b98VZDUohRCupRi14vg9rx+YV/xvJ8ysSWTUEJa65CFufgRbU7L12jSrOIuMfOebNxZfbi3NTS+TGLK+MC/oDS/JyCOWE8Qb+NTeCuM2BABr5uNJa65aNln2lWVgqtZeEahfLzSSabfNeUiy3jni/lX0kfcWUIJNICDQdjs6bdPPsroWjWjAEtRDLz4A81V0aykim2V/apBN/B8JwDzm2wkegDd84gFv4xoB1heEFF/gFro5TAp0dK1Aqgm9ZhiZo4PWG9vH3EyPNI7Tdzt7Jt5OuS9C/0Vk+WcNaR6kvKXvMGR6qlv1kTbvIcl/fWRa7Uuu3ljkhxz51GuCw4XE49okW/Gcdzmh7c7U3dXD3IAh9sXs3dsJxXWBQA/Exn2iuVyw49cTGIJ9sw5bKmsL/dP5a1f/sDq8nwCBZ/OeeWJBohOeiRjEvOElshKBuwodvV9bY5FSvqwGddvbWI4K3N6AD7eQgRnPFNniMpycFTtN59uDDqk59tX+DieCJXx9VEjERo+TfX9m0I340iL+AJNNfsEFd9B8eL6bx/VQ9m8bwDHm0YMLP8Gqbz/l64i+foBG7Sdf8bARRDgxrwYzrq8u/y3spsDh1lQXJA+ErA3ut1ej5RRRxeCRmJY+asS/3fqqIV9NY7D2v4wiNI7x/aJiWd/nPL8f8BJQoRcJvZfx5VUhttVD4gbmK86eNmkon/mX8oVs4IH+3OzC1g3IioKn7jHyF6i9W5+TkTxvipLaCkcX+6BfkrRpCW6+oc7+g1Chvyrns8/ZfkPvNtmlct6EYqGOjuPCIXDT8EQRYrrAL6KsldoVXzb2/eK0tkhvWNdlhec3uCOkvn8/8am+NV5yCEMrwQ/Ye7lbzg/YeX546b+9JrGtx98SXV8+lLZ3N9wSBmkf0PY71Sw2oKS8CS05NoTY3tfzHJZrGeEQpcvJEG+UG3QWjo2DoEvQlk4rJjDpyRSw60e0Xd7BvMF+yBARpEK8SrRHR4WBd0aCuLg1FFF7LvkiLAIY466Vri3iEnZtHv5SmoAtFlYNlS7v8djX+Qa3kZ8hW5Gkg0Bw2OlKB5XvoqN0FLU6VxOLQV+NBxJjXeDje/DtiH3wPe20WBeEnn5JWKqPLqVXr0KPvf4k6Xz1ZE21kg5DbZlXZ2WDGVsrM4rFWbOYlqsY0gst/+fmrKi/jB4KgDIESspplmNZoZ5FxGY5JNQi9LotVssfBmjryUN2fwEF12m7b3zu9as4gT4nJmdG50QQAv763aonF4z6E/LpjyPVbHZjU+Mk32nInADEj+A50ReV2eKplmeLekLK6SymX7fAkRXQhm702Q8nmFR46V1KuJQIbFU4M4nbGF7jVrSlPPK7iSRV36NKfJQY0sgdbGyDDaLxA+fj0rQrmELiVljmbmWu6DcyIleMizk2YXLuOaUAG7Gn3ZZtxiHH5qXXZP29dnv3RujnrH0M4Js+51jdmVV72g3eQyM3O8o03kWQubh8jacpwAWIfY9HUgkZI2PSrRbMcLxc1lECQLxVlPTBEIPUr1xXkTQzTgzTJkcfgG8cIFHNtx1w5Q7D28AClEwLIoLnNqE5NcY9Tn6JXx9JREyZZE6CqtcoD503v6Zianpue9qMBp5Hme5DkO6upvLcmkXca6bvXTN4bM6ABnAN6Zv/L3fWn/t3FeSCO1Zz1r84/nuPFhCuIkX/vX9KP8+Ds+ub6qn/zd7O5I7h6bFt5WHUQHKiJ/raaZuNA3Ll87ZS3PsM85XsCeP/6nlK/9Qe5JTzMjMopNMyrsenJ69cuwLDkDWwXmmzIuTI8mghZrLBJ/Jxzs9gjz2ihrGq+lRyETE54UuHUTtTB8NXBONWhgFS3ZbXNIqj3Y1fcA7cahxhxXse8uvgvs66o817qGhrduPIHw2NROoWvtWEFHeGnhktlDfPvF/bWn/4W8N0wxxN9VM4/Xtm0FwWuvLEpvlzR+klWtN4Gk1You0Qr9CkdAUWQKLy/L1AUmMVbhV7JvQe+LmqSp1FELCfkEeO0DHAF74W1fb9d/0K+i9SFF1L952hQW6Sht7bq/67Rh5P5nd0MNohZxCrCq5mI1cFSM/FqH8vNK/hK24+AKKQHgfguo1r4Ogv/zAoUEenCSV5SLQtrzt+MxiBZHjSslQolX838BKLbtH3F6TXQSSH6f2dvXCsv0mK3/WptiMMXKaVk82baShxbDTbvMq8Ja9Ntv0PkJkQ27TwOa/Jv6ibxXpmg5uG2/xrRNu5JqwhGyfC2Dx8NMYYngqDqdDwX/zr7TLXd9tTZwCUPz2KY9R26dR1EWzPU3HRgQT+vIEeqQwtL8arJHUJNgDXb3LJteeTyZT/cCcZaDcYaSzvbRNfl8TNAOFwvYuKoQxDwTcsgsEza6TRtl4+ySFRlACxtUCPCF7xw3kImna0SbcX6ZcRp73Adn3QwjllvlW9guzbDi7zKz28j091sHlXoHkoaaV/3YDyrru9y4mVylarQE7otrsk/NtCVyaE+VtzwpRveGjjbpyf8Xqjq7ysPoXVWRRXeFdvpoKq4Pew153LMdXdLaykKxJlKvHH0A/dI6QQHcBYJrwBsuCVq4j4KygAdIa93r5p4KguGC9INtibyQ7FlPBu8phhDVYWY0X4uTtR0ATKff1medVVyh74Y/z0RHy7D4qse+tZbAZz66+i3dr9MS+GtxdI4wS21PfIaRbUkm+8mSJmD+GLhrcQo2qZLs4bEKVmuzYVk7M7/4jG+pev3Hp3eIFVQrbJAz1tRDWv5M4WS8kvV4od+Y9r08NPRbjyqdWlQaGd+wy3xlVuiGMRTglBlWGPg52j0t/00U4HmaSJ+myNQJ5dL4AUkHDxXeOMFUnxf1Mo0Cb5PSd4pHZiXWKOANFrqhsGdjTpEPWNhOS+YUQJ18TpqXD152oZAjDmkZIzbzJ3EzQ1IOMCZlMzIk3lp4JXGVl7EdD+mvqtSv+gJM38KZPltXiNU12q4MC7N0cd9LW34LHwyIH8FScBdpHosz2rLY3lS4OyOKm60v7Ncr5UVBQNvSKd4mXhcLs36AXIHcCWI+e9Hrd87baOs5lEMJEOOA9IRXUgFJKTDwZ0s5DteVqPXhtAbvvSLbUhEED98x4ncs+cJG+U16p0Y4tasI91MeM/8gXnyU5SNseZjTKtZcnoCnhZIN8WUvvLfzavJzuG705MqrhJ2uvQ+M06bHVVyUe8rOGnxEc3Vn5TVEzwcZdHT8wSyGq+zlz+2Ou6eUT6VYCh35jGErY87X+OomnqdTrudPx7z2/5eD74b4bzKjsWOJg1+oTMdzyO6kbiDrjTMSzx5zb+8QKpaRaLZ6wCEMgNqGj+PRqMazgGCmT6rNNr7mTH2ctLi2C4tYdpZJsP//Z//C8Vi+Z1uJR2P4AKBqb590oKxS9Dy06tMKcUOZsp6DUtdCz/WXnvRyLebb1ZB2cTDLbR326K9W/5SJ9eAidR73mrVGI/n4C2e6NYZqYl6OQsYCCAulQzck1beRH0ClOiengn9jYxnZVlfWs/SkL4Y/+d/G8/cHANBYcQyNXunn2ExDPKMAo0JGlZyI9C5d3oCCz01t7XX95C1lW/wsbfNkxZ1aQIlmQCZLeD/6elZTU1hpEE2p9RyE37VSGkI91g3X34iYmO0azywXI277J+r72d/KHhnff69BVObw+buIV1XVs5htdAvzwasHvzPcz9shhWqCe9KihzP5CuOgGK8CAl4ROIBZN3hGwIDz+3828s7MdLcli8dMmsgeT54GL4eFDVhqNtzlLITy5A6ygZtoxFqcZ5BxAgq8SVNhHOJcQOdPJfcUQ6NCl+xI2tbrnFLx1fo3n+WVfWVHumQyTADbO64IP/GvQWjmoapNKwwjXhfHb1ekSR9qL944PrzxdXny/5/frkIQN1XqoG3EAl9vr65c9F+msevvZnquPFWKnyRE6cvkmGpICzLwfImgMgpsEZOrrsWnX/lJH+otgCs3a5D/9kO7W4dh4POgdftOvxzz+sdarcuRn7zalJoH48GnSO9/5HW/Wejzw/dUM2YmJXCGoDyZ3f95nXT5cMsoWN1XIf+s52OvHKexU4kD6GIYyvOSPxV/UWgCWYZ791n8UB0HfKDJEJoYn6a+jTkfwXCv6rrzsZsXlbGNMQ74RTT0DFCsb9Dd6l35LEMY1EaE5A1QzkTVzsIgwV9ndI74OHa7uoZmOPpfBAO/b22OFy8YJRAlqtnQGAQpNuSC5x1myjQOBgiFs09RuN4SKOFph2sXgYcnvJa2eMrEvQ42PMOHfz/wOt1uTjAw14bfjn8cx/cuiYnjRMdfE2P/OLXYN/Zdw6X7/1ETolvboz4AZnlAwyP/ObMBmrwkc3Fao9OxIEG+kGttMdf3wwa+n5XnCnBcp75uX97S9FA+ZSCiYHQz6vf9npz/dFZc41ee+tintPrWVmSpfdkkVQdheZWYSpI131Zvy2wmKcWCpV6/ZwT+KsvpztmRVHqdfDYEZVwdZmNMrvGZbaVw5oBSSHzV06xHwsi4PoJjunEjqo04GR8qDOBSH66tIe2fBwSLRRrzoz4Szf3TMUPsd/F8NQVJnyFhbtb9ftoTa/+6vK9DEjzxpY92Onst9ve8KVZ9tTi5UkYJ5C2lFrAjLPjBaa4FLLn66Udk+fEZ9d4R/ju4ly6tiWSGJg0m44Wy4BeaYnkUIU1K1QSocHSAJX5K2JpCwHZCHjkpS3lWe1vaan8b/2PlxfnwW9fLi9pb+vmy5Xp6O+U8zRmvuX9TsvvAAMONXHX6K8RCkVZvsaN3pmA4gC2+glfC5GjOho3bBGzryJQVH5dvH4ihSa56waZ1zpp4w5VJyvO+i9QEDVfEu/8423/VyCl6bxx8Ro2G/QSRWzpiqnoQ9OvRov/nNoi+lJrqQTyus4KTOJ/2/P9ttdQByLEx6vg8831324gVjL//9F0WppUdLGuH5AXORJfbgyUuejfnH0I8BzlkrKVg9XK1fBtevcGsVtiulGvx1k/r85+7nVmIb7JifMthNxEvCHc7Rf3cyzPfsZfBbidMHchx4ZQjj+Xb6J0xtMsHrPSt0wMZU260jlJ0DFiyXieUuEYg1ZT+uxlQDs7qLU7dHZuhxZpOjyC902ssOAbdvAd3sdZXvkhHewoGY4v1YtQocklhHyf4+Hp0fVx01DonRW23kaPq1s1nqBra5RK9tokSAYP/Ts+pHdE8nfG0WKbAzfhsCS9PE3hAlcjeIbs3sR6s9VoqQulQ5AEQDegHCcIANsgQKEIAtPjwrH1/wDEj/T7'
EXPECTED_SOURCE_SHA256 = 'd5dc6d40601b27dc27005f81dd19f5f186b63946532ae47b07a10f250c64478d'
source=zlib.decompress(base64.b64decode(PACKED_SOURCE))
assert hashlib.sha256(source).hexdigest()==EXPECTED_SOURCE_SHA256,'V24 evaluator source checksum mismatch'
script=Path('/content/openplaque_v24_candidates.py')
script.write_bytes(source)
compile(source,str(script),'exec')
print('Pinned V24 evaluator:',script,'SHA:',EXPECTED_SOURCE_SHA256)
print('Readable evaluator is available in Colab Files > content.')

In [ ]:
def v24(*args):
    print('RUN V24',args,flush=True)
    subprocess.run([sys.executable,str(script),*list(map(str,args))],check=True)
if RUN_SYNTHETIC_TESTS:v24('test')
v24('preflight')

In [ ]:
if RUN_PILOT:
    arguments=['run']
    if RUN_ALL_80_ONLY_IF_PILOT_PASSES:arguments+=['--full-after-pilot']
    v24(*arguments)
else:
    print('Pilot disabled, no V24 inference requested.')

In [ ]:
import json
folder=Path('/content/drive/MyDrive/OpenPlaque/OpenPlaque_Evaluation_Dataset_v0_2/evaluation_v24_vessel_candidates')
print('V24 output folder:',folder)
for name in ['run_status_v24.json','summary_v24.json','manifest_v24.json']:
    f=folder/name
    if f.exists():
        d=json.loads(f.read_text());print(name,json.dumps(d,indent=2)[:12000])
from IPython.display import HTML,display
p=folder/'OPENPLAQUE_V24_VESSEL_CANDIDATES_REPORT.html'
if p.exists():display(HTML(p.read_text()))
print('Research diagnostics only: no artery identity or full-negative ground truth established.')